# Домашнее задание 48 — Проект интеграции OpenWebUI

## Interfaces and API for LLM

В этом домашнем задании вы соберёте небольшой рабочий проект на базе OpenWebUI.

Минимально принимаемый вариант — один рабочий AI-ассистент, созданный в OpenWebUI, с подключённой Knowledge/RAG-коллекцией. Ассистент должен отвечать на вопросы по загруженным документам.

Продвинутый вариант — добавить OpenAPI tool server или MCP-инструмент через `mcpo`.

Работать можно локально через Docker Compose. Если локально запустить не получилось, можно выполнить проект в Google Colab или в другой рабочей среде, но нужно понятно показать, что именно было сделано и какой результат получился.


## Что нужно сдать

Основной формат сдачи — архив проекта.

В архиве должны быть:

1. Заполненный notebook с результатами.
2. Файлы проекта, если вы запускали локально: `docker-compose.yml`, `.env.example`, папка с документами, код tool server или MCP-настройки, если они есть.
3. Папка `screenshots/` со скриншотами результата.
4. Короткое описание запуска, если оно отличается от примера.

Если у вас не получилось запустить OpenWebUI локально, можно выполнить работу через Google Colab или другой доступный вариант. В этом случае всё равно нужно сдать архив: notebook, скриншоты, документы и краткое описание того, как вы запускали проект.

Минимальный вариант сдачи:

1. OpenWebUI запущен или показан рабочий альтернативный вариант.
2. Подключён model provider.
3. Создан один кастомный ассистент или model preset.
4. Создана Knowledge/RAG-коллекция.
5. Загружено минимум 2 документа.
6. Протестировано минимум 5 вопросов по документам.
7. Написан короткий вывод о качестве ассистента.
8. В архив добавлены скриншоты: запуск OpenWebUI, ассистент, Knowledge/RAG, загруженные документы, ответы на вопросы.

Продвинутый вариант:

1. Добавить один OpenAPI tool server и протестировать его из OpenWebUI.
2. Или добавить один MCP-инструмент через `mcpo`, например Filesystem MCP или Fetch MCP.
3. Объяснить, как инструмент расширяет возможности ассистента.


## Идея проекта

Выберите один практический use case. Не копируйте пример с занятия без изменений.

Возможные направления:

- AI-тьютор по учебным материалам.
- Ассистент по технической документации.
- Ассистент по syllabus и заданиям.
- Ассистент по внутренним документам компании.
- Research assistant для заметок и публичных веб-страниц.
- Ассистент для проверки промптов.
- Помощник по документации software-проекта.

У ассистента должна быть понятная цель и понятный целевой пользователь.


# Часть 1 — Настройка проекта

Создайте локальную папку проекта.

Пример структуры:

```text
lesson48-homework-openwebui/
├── docker-compose.yml
├── .env.example
├── docs/
│   └── your_documents_here
├── screenshots/
│   └── screenshots_here
└── optional_tool_server/
```

OpenWebUI должен открываться в браузере, например:

```text
http://localhost:3008
```

Не сдавайте настоящий API key. В архиве можно оставить `.env.example`, но не реальный `.env` с секретами.


In [2]:
# Заполните этот блок после настройки проекта.
openwebui_url = "http://localhost:3008"
model_name = ""
admin_account_created = True
model_connection_works = True

print("OpenWebUI URL:", openwebui_url)
print("Используемая модель:", model_name)
print("Admin account создан:", admin_account_created)
print("Подключение модели работает:", model_connection_works)


OpenWebUI URL: http://localhost:3008
Используемая модель: 
Admin account создан: True
Подключение модели работает: True


## Starter Docker Compose

Можно использовать этот вариант как стартовый шаблон. Если порт занят, измените его.

```yaml
services:
  open-webui:
    image: ghcr.io/open-webui/open-webui:main
    container_name: homework-open-webui
    ports:
      - "3008:8080"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - WEBUI_SECRET_KEY=${WEBUI_SECRET_KEY}
    volumes:
      - open-webui-data:/app/backend/data
    restart: unless-stopped

volumes:
  open-webui-data:
```

Пример `.env.example`:

```env
OPENAI_API_KEY=your_key_here
WEBUI_SECRET_KEY=change-this-secret-key
```

После запуска откройте OpenWebUI и создайте первый admin account.

Если локально Docker не работает, можно использовать альтернативный запуск. Главное — показать результат через notebook и скриншоты.


# Часть 2 — Создание AI-ассистента

Создайте одного ассистента, model preset или workspace agent в OpenWebUI.

Он должен включать:

- название ассистента;
- роль;
- целевого пользователя;
- system prompt;
- правила ответа по документам;
- правило для случаев, когда ответа нет в документах.

Пример структуры:

```text
Name: Course Documentation Tutor
Role: Explain uploaded course documents to beginner students.
Target user: students who need simple explanations.
Document rule: use uploaded documents as the main source.
Unknown rule: if the document does not contain the answer, say that the document does not contain enough information.
Style: clear, practical, step-by-step.
```


In [3]:
assistant_name = "Python Study Helper"
assistant_role = "Explains Python learning materials to beginner students."
target_user = "Beginner Python students."

system_prompt = """
You are a Python learning assistant.

Use the uploaded documents as the main source of truth.

Explain concepts simply and step-by-step.

If the answer is not present in the documents, clearly say:
"The uploaded documents do not contain enough information."

Be concise and educational.
"""

print("Ассистент:", assistant_name)
print("Роль:", assistant_role)
print("Целевой пользователь:", target_user)
print(system_prompt)


Ассистент: Python Study Helper
Роль: Explains Python learning materials to beginner students.
Целевой пользователь: Beginner Python students.

You are a Python learning assistant.

Use the uploaded documents as the main source of truth.

Explain concepts simply and step-by-step.

If the answer is not present in the documents, clearly say:
"The uploaded documents do not contain enough information."

Be concise and educational.



# Часть 3 — Knowledge / RAG collection

Создайте одну Knowledge/RAG-коллекцию в OpenWebUI.

Требования:

1. Загрузите минимум 2 документа.
2. Документы могут быть в формате PDF, TXT, DOCX, MD или в виде ваших собственных заметок.
3. Задайте минимум 5 вопросов.
4. Один вопрос должен быть таким, на который нет ответа в документах.
5. Проверьте, умеет ли ассистент честно отвечать, что в документах недостаточно информации.

Цель — показать, что ассистент отвечает именно по документам, а не только за счёт общих знаний модели.


In [ ]:
rag_collection_name = ""
document_names = [
    "",
    "",
]

rag_tests = [
    {
        "question": "Summarize the main ideas of the Python basics document.",
        "answer_summary": "The Python basics document provides an introduction to the Python programming language, highlighting several key points:

Overview of Python: Python is a powerful, interpreted, and interactive programming language that supports multiple paradigms such as object-oriented, procedural, and functional programming. It features clear syntax and is extensible in C or C++, making it suitable for various applications
general.md
.
Interactive Environment: The document discusses using the Python interpreter, including how to invoke it and the different modes available, such as interactive mode
index.md
.
Modules and Packages: It introduces the concept of modules, explaining the special name "main" used to identify the top-level environment of the program, along with the "main.py" file in Python packages
__main__.md
.
Learning Resources: The Python tutorial is mentioned as a starting point for beginners, and it encourages readers to explore various Python libraries in The Python Standard Library
index.md
.
The document serves as a foundational step for new learners to grasp the essentials of Python programming.",
        "grounded_in_document": True,
        "notes": "",
    },
    {
        "question": "",
        "answer_summary": "A Python dictionary is a built-in data type that stores data in key-value pairs, allowing for efficient data retrieval. Unlike sequences that are indexed by a range of numbers, dictionaries use keys to access their values. The keys can be any immutable type, such as strings or numbers. However, tuples can only be used as keys if they contain only immutable objects. Lists, being mutable, cannot be used as keys.

Dictionaries are useful for various applications where data needs to be accessed quickly using a unique key. For example, you can create a dictionary to store results by associating each key with its corresponding value",
        "grounded_in_document": True,
        "notes": "",
    },
    {
        "question": "Explain the difference between a class and an object in Python.",
        "answer_summary": "In Python, a class is a blueprint for creating objects, while an object is an instance of a class. Here’s a step-by-step explanation:

Class: A class is a template that defines the properties (attributes) and behaviors (methods) that the objects created from the class will have. It does not hold any data itself but outlines what data and functionality the objects will have when they are created.
Object: An object is an individual instantiation of a class. Each object has its own state (data) based on the attributes defined in the class. When you create an object, you allocate memory for its data and it can have its own variable values independent of other objects.
For example, if you have a class called Dog that defines attributes like breed and age, and methods like bark(), each individual dog you create from this class (like dog1 and dog2) will be objects with their own specific breed and age values but will share the same methods defined by the Dog class.

In summary, the class is the structure, while objects are the instances or realizations of that structure.",
        "grounded_in_document": True,
        "notes": "",
    },
    {
        "question": "Create 3 quiz questions about Python functions.",
        "answer_summary": "Here are three quiz questions about Python functions:

Question 1: What is the only operation that can be performed on a function object in Python?
A) Assign values
B) Call the function
C) Modify the function
D) Create a new function
Question 2: In Python, what are methods that can be called using attribute notation called?
A) Global Functions
B) Built-in Functions
C) Methods
D) Lambda Functions
Question 3: Which of the following is NOT a type of function object in Python?
A) Built-in functions
B) User-defined functions
C) Class instance methods
D) Static functions",
        "grounded_in_document": True,
        "notes": "",
    },
    {
        "question": "How does Kubernetes autoscaling work?",
        "answer_summary": "The uploaded documents do not contain enough information.",
        "grounded_in_document": False,
        "notes": "Это должен быть вопрос, на который нет ответа в загруженных документах.",
    },
]

print("RAG collection:", rag_collection_name)
print("Документы:", document_names)
rag_tests


## Обязательные типы RAG-тестов

Используйте эти пять типов тестов:

1. Вопрос на краткое резюме документа.
2. Фактический вопрос.
3. Вопрос на объяснение.
4. Вопрос на генерацию quiz или задания.
5. Вопрос вне документов.

Для каждого ответа напишите, был ли ответ основан на загруженных документах.


# Часть 4 — Короткий анализ ассистента

Напишите короткий анализ вашего RAG-ассистента.

Ответьте на вопросы:

1. Какие документы вы загрузили?
2. С чем должен помогать ассистент?
3. Какие вопросы отработали хорошо?
4. Какой вопрос был сложным?
5. Корректно ли ассистент обработал вопрос вне документов?
6. Что бы вы улучшили?


In [ ]:
rag_analysis = """
1. Загруженные документы:
Были загружены два документа: python_basics.md и oop_intro.md.
Документы содержали информацию о базовом синтаксисе Python, переменных, циклах, функциях, словарях, а также об объектно-ориентированном программировании, классах и объектах.


2. Цель ассистента:
Ассистент предназначен для помощи начинающим студентам в изучении Python.
Он отвечает на вопросы по учебным материалам, объясняет концепции простыми словами и помогает создавать quiz-вопросы по документам.

3. Вопросы, которые отработали хорошо:
Лучше всего ассистент справился с фактическими вопросами и краткими объяснениями.
Например, он корректно объяснил, что такое Python dictionary, а также разницу между class и object.
Также хорошо сработала генерация quiz-вопросов по функциям Python.

4. Сложный вопрос:
Самым сложным оказался вопрос на подробное объяснение объектно-ориентированного программирования.
Иногда ответы были слишком общими и не всегда достаточно детальными.

5. Поведение на вопросе вне документов:
Ассистент корректно обработал вопрос вне документов.
На вопрос про Kubernetes autoscaling он ответил, что в загруженных документах недостаточно информации.

6. Что можно улучшить:
Можно улучшить качество документов и добавить больше примеров кода.
Также можно настроить более качественный retrieval и добавить дополнительные инструменты, например MCP Filesystem или OpenAPI tools.
"""

print(rag_analysis)


# Часть 5 — Продвинутый вариант: OpenAPI tool server

Эта часть необязательная, но рекомендуется для более сильного проекта.

Создайте небольшой FastAPI tool server и подключите его к OpenWebUI как OpenAPI tool.

Минимальные endpoints для tool server:

```text
GET  /health
GET  /project-info
POST /your-custom-tool
```

Ваш кастомный tool должен быть полезен для ассистента. Примеры:

- проверка качества промпта;
- инструмент для статистики текста;
- планировщик задач;
- quiz helper;
- weather или другой public API tool;
- помощник по документации.

Tool должен возвращать чистый JSON.


## Starter для OpenAPI tool server

`requirements.txt`:

```text
fastapi==0.115.6
uvicorn[standard]==0.34.0
pydantic==2.10.4
requests==2.32.3
```

`Dockerfile`:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY main.py .
EXPOSE 8002
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8002"]
```

Подключение к OpenWebUI как OpenAPI:

```text
Type: OpenAPI
URL: http://tool-server:8002
OpenAPI Spec: URL → openapi.json
Auth: None
```


In [ ]:
openapi_tool_done = False
openapi_tool_tests = [
    {"tool_name": "project-info", "prompt": "", "result_summary": "", "worked": False},
    {"tool_name": "your-custom-tool", "prompt": "", "result_summary": "", "worked": False},
]

print("OpenAPI tool выполнен:", openapi_tool_done)
openapi_tool_tests


# Часть 6 — Продвинутый вариант: MCP через mcpo

Эта часть необязательная, но рекомендуется для более сильных проектов.

Добавьте одну MCP-интеграцию через `mcpo` и подключите её к OpenWebUI как OpenAPI connection.

Выберите один вариант:

## Вариант A — Filesystem MCP

Используйте его для чтения файлов только из безопасной папки, например `/workspace`.

Это полезно для локальных заметок.

## Вариант B — Fetch MCP

Используйте его для получения содержимого публичной веб-страницы.

Это полезно для документации и research notes.

Важно: объясните, к каким данным MCP-инструмент имеет доступ и к каким данным он доступа не имеет.


## Пример Filesystem MCP

`Dockerfile.filesystem`:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
RUN apt-get update && apt-get install -y nodejs npm && rm -rf /var/lib/apt/lists/*
RUN pip install --no-cache-dir mcpo
EXPOSE 8004
CMD ["sh", "-c", "mcpo --host 0.0.0.0 --port 8004 -- npx -y @modelcontextprotocol/server-filesystem /workspace"]
```

Проверка:

```text
http://localhost:8004/docs
```

Подключение в OpenWebUI:

```text
Type: OpenAPI
URL: http://mcpo-filesystem:8004
OpenAPI Spec: openapi.json
```


## Пример Fetch MCP

`Dockerfile.fetch`:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
RUN pip install --no-cache-dir mcpo uv
EXPOSE 8005
CMD ["sh", "-c", "mcpo --host 0.0.0.0 --port 8005 -- uvx mcp-server-fetch"]
```

Проверка:

```text
http://localhost:8005/docs
```

Подключение в OpenWebUI:

```text
Type: OpenAPI
URL: http://mcpo-fetch:8005
OpenAPI Spec: openapi.json
```


In [ ]:
mcp_done = False
mcp_type = ""  # filesystem или fetch
mcp_tests = [
    {"tool_name": "", "prompt": "", "result_summary": "", "worked": False},
]

print("MCP выполнен:", mcp_done)
print("Тип MCP:", mcp_type)
mcp_tests


# Часть 7 — Финальный вывод

Напишите финальный вывод.

Фокусируйтесь на своём проекте, а не на общей теории.

Ответьте:

1. Что вы собрали?
2. Какая минимальная рабочая функция есть у вашего ассистента?
3. Как RAG помогает ассистенту?
4. Добавили ли вы tools или MCP? Если да, что они добавили?
5. Какие основные ограничения есть у проекта?
6. Что бы вы улучшили в следующей версии?


In [ ]:
final_conclusion = """
В рамках проекта был собран локальный AI-ассистент на базе OpenWebUI с подключённой Knowledge/RAG-коллекцией.
Проект запускался через Docker Compose и использовал OpenAI model provider.

Минимальная рабочая функция ассистента — ответы на вопросы по загруженным документам.
Ассистент умеет:
- объяснять материалы по Python,
- делать краткие summaries,
- отвечать на фактические вопросы,
- генерировать quiz-вопросы,
- сообщать, если информации нет в документах.

RAG помогает ассистенту использовать загруженные документы как основной источник информации.
Благодаря этому ответы становятся более привязанными к конкретным материалам, а не только к общим знаниям модели.

Tools или MCP-инструменты в проект не добавлялись.
Основной фокус был сделан на создании стабильного базового RAG-ассистента и тестировании retrieval по документам.

Основные ограничения проекта:
- небольшое количество документов,
- ограниченная глубина retrieval,
- ответы иногда были слишком общими,
- отсутствует автоматическая синхронизация локальных файлов с Knowledge Base.

В следующей версии проекта можно улучшить:
- качество и объём документов,
- настройки chunking и retrieval,
- добавить MCP Filesystem или OpenAPI tools,
- реализовать автоматическую загрузку документов,
- добавить поддержку нескольких специализированных ассистентов.
"""
print(final_conclusion)

# Финальный checklist перед сдачей

Минимальный checklist:

- [ ] OpenWebUI открывается локально или показан рабочий альтернативный вариант.
- [ ] Model connection работает.
- [ ] Создан один custom assistant или model preset.
- [ ] Создана Knowledge/RAG collection.
- [ ] Загружено минимум 2 документа.
- [ ] Протестировано минимум 5 RAG-вопросов.
- [ ] Протестирован один вопрос вне документов.
- [ ] Написан короткий RAG-анализ.
- [ ] Написан финальный вывод.
- [ ] В архив добавлены скриншоты результата.

Продвинутый checklist:

- [ ] OpenAPI tool server подключён и протестирован.
- [ ] MCP через `mcpo` подключён и протестирован.
- [ ] Объяснено, что добавил tool или MCP.

Что сдавать:

1. Архив `.zip` или `.rar` с проектом.
2. Заполненный notebook.
3. Скриншоты в папке `screenshots/`.
4. Документы, которые использовались для Knowledge/RAG, если их можно сдавать.
5. Файлы запуска: `docker-compose.yml`, `.env.example`, `requirements.txt`, `Dockerfile`, `main.py` — если они использовались.

Если работа выполнена в Google Colab, сдайте архив с notebook, скриншотами, документами и кратким описанием запуска.

Важно: не сдавайте настоящие API keys, passwords, tokens и другие секреты.
